# 04. LoRA 튜닝 - 도메인 특화 모델 학습

노션 문서를 기반으로 Q&A 데이터셋을 생성하고,
LoRA(Low-Rank Adaptation) 기법으로 로컬 LLM을 도메인에 특화시킵니다.

## LoRA 소개
- 전체 모델 파라미터 대신 작은 어댑터만 학습
- 메모리 효율적 (8GB VRAM으로 가능)
- 원본 모델 보존 + 도메인 지식 추가

## 학습 흐름
```
1. Q&A 데이터셋 생성 (노션 문서 → GPT/Claude로 Q&A 쌍 생성)
2. 데이터셋 포맷팅 (Alpaca 형식)
3. QLoRA 설정 (4bit 양자화 + LoRA)
4. 학습 실행
5. 튜닝된 모델 평가
```

---
## 1. 필요 패키지 설치

```bash
pip install transformers torch accelerate bitsandbytes peft datasets trl
```

In [ ]:
import os
import sys
import json
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv()

# GPU 확인
try:
    import torch
    if torch.cuda.is_available():
        print(f"✓ CUDA 사용 가능")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")
        GPU_AVAILABLE = True
    else:
        print("✗ CUDA 사용 불가 - CPU 모드")
        GPU_AVAILABLE = False
except ImportError:
    print("✗ PyTorch 미설치")
    GPU_AVAILABLE = False

---
## 2. Q&A 데이터셋 생성

노션 문서에서 Q&A 쌍을 자동으로 생성합니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Q&A 생성용 프롬프트
QA_GENERATION_PROMPT = """다음 개발 문서를 읽고, 이 문서의 내용을 기반으로 Q&A 쌍을 3개 생성해주세요.

요구사항:
1. 질문은 실제 개발자가 할 법한 자연스러운 질문이어야 합니다.
2. 답변은 문서의 내용을 기반으로 정확하고 실용적이어야 합니다.
3. 코드 예시가 있다면 답변에 포함해주세요.

문서:
{document}

다음 JSON 형식으로 출력해주세요:
[
  {{
    "question": "질문 내용",
    "answer": "답변 내용"
  }}
]"""

qa_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
qa_prompt = ChatPromptTemplate.from_template(QA_GENERATION_PROMPT)
qa_chain = qa_prompt | qa_llm | JsonOutputParser()

print("Q&A 생성 체인 준비 완료")

In [ ]:
from src.loaders.notion_loader import create_sample_documents

# 샘플 문서로 Q&A 생성
sample_docs = create_sample_documents()

all_qa_pairs = []

for doc in sample_docs:
    title = doc.metadata.get("title", "Unknown")
    print(f"\n문서 처리 중: {title}")

    try:
        qa_pairs = qa_chain.invoke({"document": doc.page_content})
        for qa in qa_pairs:
            qa["source"] = title
        all_qa_pairs.extend(qa_pairs)
        print(f"  → {len(qa_pairs)}개 Q&A 쌍 생성")
    except Exception as e:
        print(f"  → 생성 실패: {e}")

print(f"\n총 {len(all_qa_pairs)}개 Q&A 쌍 생성 완료")

In [ ]:
# 생성된 Q&A 샘플 확인
for i, qa in enumerate(all_qa_pairs[:3], 1):
    print(f"\n[Q&A {i}] (출처: {qa['source']})")
    print(f"Q: {qa['question']}")
    print(f"A: {qa['answer'][:200]}..." if len(qa['answer']) > 200 else f"A: {qa['answer']}")

---
## 3. 데이터셋 포맷팅 (Alpaca 형식)

LoRA 학습에 사용할 Alpaca 형식으로 변환합니다.

```json
{
  "instruction": "질문",
  "input": "",
  "output": "답변"
}
```

In [ ]:
def convert_to_alpaca(qa_pairs):
    """Q&A 쌍을 Alpaca 형식으로 변환"""
    alpaca_data = []
    for qa in qa_pairs:
        alpaca_data.append({
            "instruction": qa["question"],
            "input": "",
            "output": qa["answer"]
        })
    return alpaca_data

alpaca_dataset = convert_to_alpaca(all_qa_pairs)

# 데이터셋 저장
os.makedirs("../data/processed", exist_ok=True)
dataset_path = "../data/processed/alpaca_dataset.json"

with open(dataset_path, "w", encoding="utf-8") as f:
    json.dump(alpaca_dataset, f, ensure_ascii=False, indent=2)

print(f"데이터셋 저장: {dataset_path}")
print(f"총 {len(alpaca_dataset)}개 샘플")

In [ ]:
# 저장된 데이터셋 확인
print(json.dumps(alpaca_dataset[0], indent=2, ensure_ascii=False))

---
## 4. QLoRA 설정

4bit 양자화 + LoRA 조합으로 메모리 효율적인 학습을 설정합니다.

In [ ]:
# 패키지 import (실제 학습 시 주석 해제)
if GPU_AVAILABLE:
    try:
        from transformers import (
            AutoModelForCausalLM,
            AutoTokenizer,
            BitsAndBytesConfig,
            TrainingArguments
        )
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
        from datasets import load_dataset
        from trl import SFTTrainer

        print("✓ 모든 패키지 로드 완료")
        TRAINING_AVAILABLE = True
    except ImportError as e:
        print(f"✗ 패키지 미설치: {e}")
        print("  pip install transformers peft datasets trl bitsandbytes")
        TRAINING_AVAILABLE = False
else:
    TRAINING_AVAILABLE = False

In [ ]:
# QLoRA 설정 (8GB VRAM 최적화)
if TRAINING_AVAILABLE:
    # 4bit 양자화 설정
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True  # 추가 메모리 절약
    )

    # LoRA 설정
    lora_config = LoraConfig(
        r=8,                           # rank (낮게 유지하여 메모리 절약)
        lora_alpha=16,                 # alpha = r * 2
        target_modules=["q_proj", "v_proj"],  # 어텐션 레이어만
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    print("QLoRA 설정 완료")
    print(f"  - rank: {lora_config.r}")
    print(f"  - alpha: {lora_config.lora_alpha}")
    print(f"  - target_modules: {lora_config.target_modules}")

In [ ]:
# 모델 로드
if TRAINING_AVAILABLE:
    MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

    print(f"모델 로딩 중: {MODEL_NAME}")
    print("(첫 실행 시 ~5GB 다운로드가 필요합니다)")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    # LoRA를 위한 모델 준비
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)

    # 학습 가능한 파라미터 확인
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"\n모델 로드 완료")
    print(f"  전체 파라미터: {total_params:,}")
    print(f"  학습 파라미터: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")

---
## 5. 학습 실행

In [ ]:
# 데이터셋 로드
if TRAINING_AVAILABLE:
    dataset = load_dataset("json", data_files=dataset_path, split="train")
    print(f"데이터셋 로드: {len(dataset)}개 샘플")

In [ ]:
# 프롬프트 포맷팅 함수
def formatting_prompts_func(example):
    """Alpaca 형식을 모델 입력 형식으로 변환"""
    output_texts = []
    for instruction, input_text, output in zip(
        example["instruction"],
        example["input"],
        example["output"]
    ):
        if input_text:
            text = f"""### 질문:\n{instruction}\n\n### 추가 정보:\n{input_text}\n\n### 답변:\n{output}"""
        else:
            text = f"""### 질문:\n{instruction}\n\n### 답변:\n{output}"""
        output_texts.append(text)
    return output_texts

In [ ]:
# 학습 인자 설정 (8GB VRAM 최적화)
if TRAINING_AVAILABLE:
    training_args = TrainingArguments(
        output_dir="../models/lora-adapter",
        num_train_epochs=3,
        per_device_train_batch_size=1,       # 배치 사이즈 1 (VRAM 절약)
        gradient_accumulation_steps=4,        # 누적으로 effective batch size = 4
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=10,
        save_steps=50,
        fp16=True,                           # 혼합 정밀도 학습
        optim="paged_adamw_8bit",           # 메모리 효율적 옵티마이저
        max_grad_norm=0.3,
        report_to="none"                    # wandb 비활성화
    )

    print("학습 인자 설정 완료")

In [ ]:
# 트레이너 설정 및 학습 실행
if TRAINING_AVAILABLE:
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        peft_config=lora_config,
        tokenizer=tokenizer,
        args=training_args,
        formatting_func=formatting_prompts_func,
        max_seq_length=512
    )

    print("학습 시작...")
    trainer.train()
    print("학습 완료!")

In [ ]:
# LoRA 어댑터 저장
if TRAINING_AVAILABLE:
    adapter_path = "../models/lora-adapter"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"LoRA 어댑터 저장: {adapter_path}")

---
## 6. 튜닝된 모델 평가

In [ ]:
# 튜닝된 모델 로드
if TRAINING_AVAILABLE:
    from peft import PeftModel

    # 베이스 모델 로드
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )

    # LoRA 어댑터 적용
    tuned_model = PeftModel.from_pretrained(base_model, "../models/lora-adapter")

    print("튜닝된 모델 로드 완료")

In [ ]:
# 평가 함수
def generate_response(model, tokenizer, question, max_new_tokens=256):
    """질문에 대한 답변 생성"""
    prompt = f"### 질문:\n{question}\n\n### 답변:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 답변 부분만 추출
    if "### 답변:" in response:
        response = response.split("### 답변:")[1].strip()

    return response

In [ ]:
# 튜닝 전/후 비교
if TRAINING_AVAILABLE:
    test_questions = [
        "Spring Boot에서 JPA 설정은 어떻게 하나요?",
        "REST API에서 리소스 생성 성공 시 어떤 HTTP 상태 코드를 사용해야 하나요?",
        "Spring Security에서 특정 URL을 인증 없이 접근하게 하려면?"
    ]

    for q in test_questions:
        print(f"\n{'='*60}")
        print(f"질문: {q}")
        print(f"{'='*60}")

        # 튜닝 전 (베이스 모델)
        print("\n[베이스 모델]")
        base_answer = generate_response(base_model, tokenizer, q)
        print(base_answer[:300] + "..." if len(base_answer) > 300 else base_answer)

        # 튜닝 후
        print("\n[튜닝된 모델]")
        tuned_answer = generate_response(tuned_model, tokenizer, q)
        print(tuned_answer[:300] + "..." if len(tuned_answer) > 300 else tuned_answer)

---
## 7. 튜닝된 모델을 RAG에 적용

In [ ]:
# LangChain에 튜닝된 모델 연동
if TRAINING_AVAILABLE:
    from langchain_community.llms import HuggingFacePipeline
    from transformers import pipeline

    # 파이프라인 생성
    pipe = pipeline(
        "text-generation",
        model=tuned_model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        do_sample=False
    )

    # LangChain LLM으로 래핑
    tuned_llm = HuggingFacePipeline(pipeline=pipe)

    print("튜닝된 모델 LangChain 연동 완료")

In [ ]:
# 튜닝된 모델로 RAG 체인 구성
if TRAINING_AVAILABLE:
    from src.embeddings.embedding_manager import EmbeddingManager
    from src.vectorstore.chroma_store import ChromaVectorStore
    from src.chains.rag_chain import RAGChain

    # 기존 벡터 스토어 사용
    embeddings = EmbeddingManager(provider="openai").embeddings
    vector_store = ChromaVectorStore(
        embeddings=embeddings,
        persist_directory="../data/chroma_sample",
        collection_name="sample_docs"
    )

    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    # 튜닝된 모델로 RAG 체인
    tuned_rag = RAGChain(llm=tuned_llm, retriever=retriever)

    # 테스트
    question = "Spring Boot에서 JPA 설정 방법을 알려주세요."
    answer = tuned_rag.invoke(question)
    print(f"질문: {question}")
    print(f"\n답변: {answer}")

---
## 정리

### LoRA 튜닝 체크리스트

- [ ] Q&A 데이터셋 생성 (100-500개 샘플 권장)
- [ ] Alpaca 형식으로 포맷팅
- [ ] QLoRA 설정 (4bit + LoRA)
- [ ] 학습 실행 (3-5 에포크)
- [ ] 튜닝 전/후 품질 비교

### 성능 향상을 위한 팁

1. **데이터 품질**: Q&A 쌍의 품질이 가장 중요
2. **데이터 다양성**: 다양한 질문 유형 포함
3. **하이퍼파라미터**: 
   - rank가 높을수록 표현력 증가 (메모리도 증가)
   - learning rate는 2e-4 ~ 5e-5 범위에서 조정
4. **과적합 방지**: 검증 세트로 조기 종료 고려

### 다음 단계

- 실제 노션 문서로 더 많은 Q&A 데이터 생성
- 검증 세트 분리 및 평가 지표 추가
- 프로덕션 배포를 위한 모델 최적화 (GGUF 변환 등)